In [1]:
# load ckpt : gpt2 small
from src.gapt import GPTConfig, GPT
import torch

ckpt_path = "ckpt/iblm-fineweb10B-gpt2-small-20k.pt"
modgpt_config = GPTConfig.prior(name="small")
modgpt_model = GPT(modgpt_config)

# Load ckpt 
ckpt = torch.load(ckpt_path, map_location="cpu")
state_dict = ckpt["model"]
# Fix for torch.compile: strip "_orig_mod." prefix if present
wanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(wanted_prefix):
        state_dict[k[len(wanted_prefix):]] = state_dict.pop(k)
modgpt_model.load_state_dict(state_dict)


<All keys matched successfully>

In [2]:
# hf_config = CustomGPTConfig(modgpt_config)
from src.modeling_custom_gpt import CustomGPTConfig, CustomGPTModel, port_weights

hf_config = CustomGPTConfig(
    vocab_size=modgpt_config.vocab_size,
    n_embd=modgpt_config.n_embd,
    n_layer=modgpt_config.n_layer,
    n_head=modgpt_config.n_head,
    flex_kernel_options=modgpt_config.flex_kernel_options
)
hf_model = CustomGPTModel(hf_config)

mapped_state = port_weights(modgpt_model.state_dict(), hf_model.state_dict(), hf_config)
hf_model.load_state_dict(mapped_state, strict=True)

<All keys matched successfully>

In [3]:
# Upload model checkpoint to HuggingFace Hub

from huggingface_hub import HfApi, login, create_repo, ModelCard
from transformers import AutoTokenizer

hf_usr_name = "Ksgk-fy"
hf_repo_name = "iblm"
full_repo = f"{hf_usr_name}/{hf_repo_name}"

# 2. Create repo if doesn't exist
api = HfApi()
try:
    api.create_repo(full_repo, repo_type="model", exist_ok=True)
except Exception as e:
    print("Repo may already exist or there was an error. Proceeding...")
    
local_dir = "./hf_ckpt"
# This will write config.json WITH the auto_map field
hf_model.save_pretrained(local_dir) 

# Upload EVERYTHING (config.json, model.safetensors, modeling_custom_gpt.py)
from huggingface_hub import upload_folder

# First, copy the python file into the save directory so upload_folder grabs it
import shutil
shutil.copy("src/modeling_custom_gpt.py", f"{local_dir}/modeling_custom_gpt.py")

upload_folder(
    repo_id=full_repo,
    folder_path=local_dir,
    repo_type="model",
    commit_message="Fix custom model loading"
)

# print(f"Model uploaded to https://huggingface.co/{full_repo}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Ksgk-fy/iblm/commit/92b8caab38c8dfe582ac46d04951d8594bf9554e', commit_message='Fix custom model loading', commit_description='', oid='92b8caab38c8dfe582ac46d04951d8594bf9554e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ksgk-fy/iblm', endpoint='https://huggingface.co', repo_type='model', repo_id='Ksgk-fy/iblm'), pr_revision=None, pr_num=None)

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Ksgk-fy/iblm", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("gpt2") # Assuming GPT2 tokenizer

input_text = "Once upon a time"
input_ids = tokenizer.encode(input_text, return_tensors="pt")

with torch.no_grad():
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=50, 
        do_sample=True, 
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

Once upon a time tohipTuesdayhipTuesdayhipTuesdayhiphiphip,, issuesic, or
TuesdayPHSundayTuesdaySunday- dramaTuesdayP2,
SundayHhipP beehip,), alikehipic, servicehip of,
PPSunday
